This notebook will utilize the GSTools and PyKrige libraries to implement Ordinary Kriging and Universal Kriging for point cloud data. Data will be interpolated onto a grid with horizontal spatial resolution of your chosing. This grid may be clipped by a boundary shapefile if you have one.

In [ ]:
%load_ext autoreload
%autoreload 2
import os, sys
sys.path.insert(0, os.path.abspath('src'))
import geopandas as gpd
import pandas as pd
from shapely import wkt
import gstools as gs

from OK import OK_AIC
from UK import UK_AIC
from RK import RK_AIC

In [ ]:
boundary_gdf = gpd.read_file(f'{os.path.abspath('raw_data')}/boundary.shp')
pts_df = pd.read_csv(f'{os.path.abspath("raw_data")}/sampled_pts.csv')
pts_df['geometry'] = pts_df['geometry'].apply(wkt.loads)
example_pts = gpd.GeoDataFrame(pts_df, geometry='geometry', crs=boundary_gdf.crs)

In [ ]:
# First, extract the x, y, z data from your dataset

# this is for my example data in the raw_data directory

x = example_pts.xLocation.values
y = example_pts.yLocation.values
z = example_pts.Z_use.values

In [ ]:
# next establish what variogram models you wish to test with GSTools

# some typical ones I default to the below, since some GSTools models are not implemented within PyKrige

vario_models = {
    "Gaussian": gs.Gaussian,
    "Exponential": gs.Exponential,
    "Matern": gs.Matern,
    "Stable": gs.Stable,
    "Rational": gs.Rational,
    "Circular": gs.Circular,
    "Spherical": gs.Spherical,
    "SuperSpherical": gs.SuperSpherical,
    "JBessel": gs.JBessel,
}

In [ ]:
# you can change these to save whereever you like
ok_pred_out_path = f'{os.path.abspath('model_outputs')}/OK_test_predictions.tif'      
ok_var_out_path = f'{os.path.abspath('model_outputs')}/OK_test_variances.tif'

estimator = 'cressie'   # or 'matheron'. Two different approaches to weighted least square solution for automatic variogram fitting. Cressie is more robust to outliers

pred_grid_resolution = 10.0     # this will be in the unit of your CRS. For the example data in raw_data this is feet. You can adjust to be the size you wish, preferably a whole number

ok = OK_AIC(
    vario_models, x, y, z, boundary_gdf, 
    ok_pred_out_path, ok_var_out_path, 
    vario_estimator = estimator, 
    grid_res = pred_grid_resolution, 
    plot_outputs = True
    )

pred_path, var_path = ok.generate()

In [ ]:
uk_trend = "regional_linear" # or None. There are more options, but I have not explored or developed those within the UK_AIC class yet

uk = UK_AIC(
    vario_models, x, y, z, boundary_gdf,
    'model_outputs/UK_test_predictions.tif', 'model_outputs/UK_test_variances.tif',
    trend_model = uk_trend,
    vario_estimator = estimator,
    grid_res=10.0,
    plot_outputs=True,
)

pred_path, var_path = uk.generate()

In [ ]:
# I employed something like this for my presentation at hydroreads. Though instead of sklearn regression models, I was using Verde Green's function tension splines

rk = RK_AIC(
    x, y, z, boundary_gdf,
    pred_out_path="model_outputs/RK_test_predictions.tif",
    var_out_path="model_outputs/RK_test_variances.tif",
    regression_model="svr",
    regression_model_params=(),         # you can head to sklearn to look at regression models and parameters for input as a dictionary. e.g., {'random_state': 42, 'n_estiamtors': 300}
    vario_models=vario_models,          # same dictionary you use for OK/UK
    vario_estimator=estimator,          # same as OK/UK
    grid_res=pred_grid_resolution,
    plot_outputs=True,
)
pred_path, var_path = rk.generate()